# **_Industrial Vision SLM Benchmark_**
## **_Notebook 02 — Construction du corpus documentaire_**

### **Objectif**

Construire un corpus de documentation technique fiable et
traçable pour alimenter le futur pipeline RAG.

### **Périmètre**

Le corpus couvrira cinq domaines :

1. Filtrage d'images.
2. Morphologie mathématique.
3. Segmentation.
4. Détection et analyse de contours.
5. Inspection industrielle.

### **Méthodologie**

- Sélectionner les sources documentaires.
- Enregistrer leurs métadonnées.
- Récupérer les documents.
- Nettoyer les contenus.
- Découper les documents en passages (*chunks*).
- Contrôler la qualité du corpus.

### **Contraintes**

- Privilégier la documentation officielle OpenCV.
- Conserver l'origine de chaque document.
- Exclure les questions, corrigés et annotations du benchmark.
- Ne pas modifier la baseline déjà finalisée.

## **1. Initialisation du corpus documentaire**

**Objectif :** préparer une organisation reproductible du corpus.

**Résultats attendus :**
- Dossiers raw, processed et chunks.
- Registre source_registry.json.
- Cinq premières sources enregistrées.

**Point de vigilance :** aucune documentation n'est encore téléchargée.

In [ ]:
import json
from pathlib import Path

# 1. Racine du dépôt
current_dir = Path.cwd().resolve()

candidate_roots = [current_dir, current_dir.parent]
project_roots = [path for path in candidate_roots if (
    (path / "README.md").is_file() and (path / "data" / "evaluation").is_dir()
    )
]

assert len(project_roots) == 1, "Racine du projet introuvable. Vérifier le dossier de travail du notebook."

project_dir = project_roots[0]
print("[OK] Projet trouvé :", project_dir)

# 2. Définition des dossiers du corpus
corpus_dir = project_dir / "data" / "corpus"
raw_dir = corpus_dir / "raw"
processed_dir = corpus_dir / "processed"
chunks_dir = corpus_dir / "chunks"

for directory in [raw_dir, processed_dir, chunks_dir]:
    directory.mkdir(parents=True, exist_ok=True)
    print("[OK] Dossier :", directory.relative_to(project_dir))

# 3. Definition des 5 premières sources officielles
sources = [
    {
        "source_id": "opencv_filtering",
        "topic": "filtering",
        "title": "Smoothing Images",
        "source_url": "https://docs.opencv.org/4.13.0/d4/d13/tutorial_py_filtering.html",
        "opencv_version": "4.13.0",
        "status": "planned"
    },
    {
        "source_id": "opencv_morphology",
        "topic": "morphology",
        "title": "Morphological Transformations",
        "source_url": "https://docs.opencv.org/4.13.0/d9/d61/tutorial_py_morphological_ops.html",
        "opencv_version": "4.13.0",
        "status": "planned"
    },
    {
        "source_id": "opencv_thresholding",
        "topic": "segmentation",
        "title": "Image Thresholding",
        "source_url": "https://docs.opencv.org/4.13.0/d7/d4d/tutorial_py_thresholding.html",
        "opencv_version": "4.13.0",
        "status": "planned"
    },
    {
        "source_id": "opencv_contours",
        "topic": "edges",
        "title": "Contours Hierarchy",
        "source_url": "https://docs.opencv.org/4.13.0/d9/d8b/tutorial_py_contours_hierarchy.html",
        "opencv_version": "4.13.0",
        "status": "planned"
    },
    {
        "source_id": "opencv_template_matching",
        "topic": "industrial_inspection",
        "title": "Template Matching",
        "source_url": "https://docs.opencv.org/4.13.0/d4/dc6/tutorial_py_template_matching.html",
        "opencv_version": "4.13.0",
        "status": "planned"
    }
]

# 4. Définition du fichier du registre
registry_path = corpus_dir / "source_registry.json"

# Eviter d'écraser un registre existant
if registry_path.exists():
    with registry_path.open("r", encoding="utf-8") as file:
        registry = json.load(file)

    print("[INFO] Registre existant conservé.")
else:
    registry = {
        "corpus_name": "industrial_vision_opencv",
        "opencv_version": "4.13.0",
        "sources": sources
    }

    with registry_path.open("w", encoding="utf-8") as file:
        json.dump(registry, file, ensure_ascii=False, indent=2)

    print("[OK] Nouveau registre créé.")

# 5. Vérif du registre
source_list = registry["sources"]
source_ids = [source["source_id"] for source in source_list]

assert len(source_list) >= 5
assert len(set(source_ids)) == len(source_ids)

print("\n=== Sources documentaires ===")

for source in source_list:
    print(f"- {source['topic']} : {source['title']} [{source['status']}]")

print("\nNombre de sources :", len(source_list))
print("Registre :", registry_path.relative_to(project_dir))
print("[OK] Initialisation du corpus terminée.")

## **2. Sélection et validation des sources documentaires**

### **Objectif**

Vérifier la cohérence du registre documentaire et compléter
les domaines insuffisamment couverts.

### **Problème identifié**

La source consacrée à la hiérarchie des contours ne couvre
pas à elle seule la détection de contours avec Canny ou
le calcul des gradients d'image.

### **Méthode**

1. Ajouter deux documents officiels au domaine `edges`.
2. Vérifier les identifiants et les métadonnées.
3. Contrôler la structure des URLs.
4. Sauvegarder le registre enrichi.

### **Résultat attendu**

Sept sources enregistrées, réparties sur cinq domaines.

**Remarque :** cette étape valide les métadonnées du registre.
Les téléchargements et vérifications HTTP seront effectués
dans la section 3.

In [ ]:
from urllib.parse import urlparse

# 1. Chargement du registre
with registry_path.open("r", encoding="utf-8") as file:
    registry = json.load(file)

print("[OK] Registre chargé.")

# 2. Nouvelles sources à ajouter
additional_sources = [
    {
        "source_id": "opencv_canny",
        "topic": "edges",
        "title": "Canny Edge Detection",
        "source_url": "https://docs.opencv.org/4.13.0/da/d22/tutorial_py_canny.html",
        "opencv_version": "4.13.0",
        "status": "planned"
    },
    {
        "source_id": "opencv_gradients",
        "topic": "edges",
        "title": "Image Gradients",
        "source_url": "https://docs.opencv.org/4.13.0/d5/d0f/tutorial_py_gradients.html",
        "opencv_version": "4.13.0",
        "status": "planned"
    }
]
# 3. Ajout des sources absentes
existing_ids = {source["source_id"] for source in registry['sources']}
added_count = 0

for source in additional_sources:
    if source['source_id'] not in existing_ids:
        registry['sources'].append(source)
        existing_ids.add(source["source_id"])
        added_count += 1

print("[OK] Nouvelles sources ajoutées :", added_count)

# 4. Vérif des métadonnées
required_fields = {"source_id", "topic", "title", "source_url", "opencv_version", "status"}
expected_topics = {"filtering", "morphology", "segmentation", "edges", "industrial_inspection"}

source_ids = []

for source in registry["sources"]:
    # Vérif de la présence des champs obligatoires
    assert required_fields.issubset(source)

    # Vérif de la version annoncée
    assert source['opencv_version'] == "4.13.0"

    # Vérif du domaine
    assert source['topic'] in expected_topics

    # Vérif de la structure de l'URL officielle
    parsed_url = urlparse(source['source_url'])

    assert parsed_url.scheme == "https"
    assert parsed_url.hostname == "docs.opencv.org"
    assert parsed_url.path.startswith("/4.13.0/")

    source_ids.append(source["source_id"])

# 5. Vérif de l'unicité des identifiants
assert len(source_ids) == len(set(source_ids))

# Vérif de la présence de chaque domaine
present_topics = {source['topic'] for source in registry["sources"]}
assert present_topics == expected_topics

print("[OK] Métadonnées valides.")
print("[OK] Identifiants uniques.")
print("[OK] Cinq domaines représentés.")

# 6. Sauvegarde 'uniquement s'il y a modifs)
if added_count > 0:
    with registry_path.open("w", encoding="utf-8") as file:
        json.dump(registry, file, ensure_ascii=False, indent=2)

    print("[OK] Registre enrichi et sauvegardé.")
else:
    print("[INFO] Registre déjà à jour.")

# 7. Affichage de la répartition des sources
print("\n=== Répartition par domaine ===")

for topic in sorted(expected_topics):
    count = sum(source['topic'] == topic for source in registry['sources'])
    print(f"{topic} : {count} source(s)")

print("\nTotal :", len(registry['sources']))
print("[OK] Section 2 terminée.")


## **3. Téléchargement contrôlé des documents**

### **Objectif général**

Récupérer les documents techniques OpenCV enregistrés dans
notre registre afin de constituer le corpus documentaire brut
du futur système RAG.

### **Nécessité de cette étape**

Notre registre contient actuellement sept sources, mais
leurs métadonnées ne suffisent pas pour alimenter un RAG.

Nous devons récupérer leur contenu, vérifier leur intégrité
et conserver des copies locales avant de procéder à
l'extraction et au nettoyage du texte.

### **Méthodologie**

Le téléchargement sera réalisé en deux étapes :

1. **Section 3.1 — Premier téléchargement contrôlé**
   - Télécharger une seule page OpenCV.
   - Vérifier la réponse HTTP.
   - Contrôler le contenu HTML.
   - Enregistrer le fichier brut.
   - Calculer son empreinte SHA-256.
   - Mettre à jour le registre documentaire.

2. **Section 3.2 — Téléchargement automatisé du corpus**
   - Parcourir les sept sources enregistrées.
   - Réutiliser les fichiers déjà présents.
   - Télécharger les documents manquants.
   - Contrôler les empreintes SHA-256.
   - Sauvegarder les métadonnées après chaque document.

### **Résultat attendu**

À l'issue de cette section :

- Sept documents HTML seront disponibles localement.
- Chaque document possédera un identifiant unique.
- Chaque document disposera d'une empreinte SHA-256.
- Les sept sources auront le statut `downloaded`.
- Le registre permettra de retrouver l'origine des fichiers.

### **Contraintes**

- Utiliser la documentation officielle OpenCV 4.13.0.
- Ne pas inclure les questions et corrigés du benchmark.
- Ne pas modifier les résultats de la baseline.
- Ne pas télécharger plusieurs fois un fichier local valide.
- Conserver les fichiers bruts séparément des textes nettoyés.

### _3.1. Premier téléchargement contrôlé_

#### **Objectif**

Télécharger un premier document depuis la documentation
officielle OpenCV afin de comprendre chaque opération
nécessaire à la récupération fiable d'une page HTML.

#### **Document sélectionné**

- **Titre :** Smoothing Images
- **Domaine :** Filtering
- **Source :** documentation OpenCV 4.13.0
- **Identifiant :** `opencv_filtering`
- **Destination :** `data/corpus/raw/opencv_filtering.html`

#### **Étapes de l'expérience**

1. Charger le registre documentaire.
2. Rechercher la source `opencv_filtering`.
3. Vérifier si une copie locale existe déjà.
4. Envoyer une requête HTTP avec `requests` si nécessaire.
5. Vérifier le code de réponse et le type de contenu.
6. Contrôler la structure HTML du document.
7. Sauvegarder le fichier brut.
8. Calculer son empreinte SHA-256.
9. Mettre à jour les métadonnées du registre.

#### **Concepts étudiés**

**Requête HTTP :**
Communication entre notre programme Python et le serveur
hébergeant la documentation.

**Code HTTP 200 :**
Indique que le serveur a traité la requête avec succès.

**Fichier brut :**
Copie du document récupéré, avant toute transformation
ou extraction de texte.

**Empreinte SHA-256 :**
Signature numérique permettant de vérifier l'identité
du contenu d'un fichier et de détecter ses modifications.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone

import requests

# 1. Rechargement du registre depuis le disque
with registry_path.open("r", encoding="utf-8") as file:
    registry = json.load(file)

# 2. Sélection de la documentation sur le filtrage
source = next((item for item in registry['sources'] if item["source_id"] == "opencv_filtering"), None)
assert source is not None, "Source introuvable dans le registre."

print("[INFO] Document :", source['title'])
print("[INFO] URL :", source["source_url"])

# 3. Définition du fichier HTML de destination
raw_path = raw_dir / f"{source['source_id']}.html"

# 4. Téléchargement uniquement si le fichier n'existe pas
if raw_path.exists():
    # Réutiliser le copie locale lors d'une nouvelle exécution
    document = raw_path.read_bytes()
    print("[INFO] Fichier local déjà présent.")
else:
    # Effectuer la requête avec un délai maximal d'attente
    try:
        response = requests.get(
            source["source_url"],
            headers={
                "User-Agent": "IndustrialVisionSLMBenchmark/1.0 (educational project)"
            },
            timeout=20
        )

        # Déclencher une erreur si le serveur refuse la requête
        response.raise_for_status()

    except requests.RequestException as error:
        print("[ERREUR] Echec du téléchargement :", error)
        raise

    # Vérif que le serveur annonce un doc HTML
    content_type = response.headers.get("Content-Type", "").lower()
    assert "text/html" in content_type, f"Type de contenu inattendu : {content_type}"

    # Vérif que l'URL finale reste sur la documentation prévue
    assert response.url.startswith("https://docs.opencv.org/4.13.0/"),  "Redirection inattendue."

    document = response.content
    print("[OK] Reponse HTTP :", response.status_code)

# 5. Vérif que le contenu ressemble à une page OpenCV
assert len(document) > 1000, "Document anormalement court"
assert b"<html" in document[:2000].lower(), "Structure HTML non détectée."
assert b"opencv" in document.lower(), "Mention OpenCV absente du document."

print("[OK] Contenu HTML vérifié.")

# 6. Enregistrement du doc brut, si nécessaire
if not raw_path.exists():
    raw_path.write_bytes(document)
    print("[OK] Document sauvegardé.")

# 7. Calcul de l'empreinte SHA-256
checksum = hashlib.sha256(document).hexdigest()
print("[OK] SHA-256 :", checksum)

# 8. Mise à jour des métadonnées du document
source['status'] = "downloaded"
source['raw_path'] = raw_path.relative_to(project_dir).as_posix()
source['sha_256'] = checksum

source.setdefault("downloaded_at", datetime.now(timezone.utc).isoformat(timespec="seconds"))

# 9. Sauvegarde du registre mis à jour
with registry_path.open("w", encoding="utf-8") as file:
    json.dump(registry, file, ensure_ascii=False, indent=2)

print("[OK] Registre mis à jour.")
print("\n=== Résultat ===")

print("Document :", source["title"])
print("Statut :", source["status"])
print("Fichier :", source["raw_path"])
print("Taille :", len(document), "octets")
print("[OK] Premier téléchargement terminé.")

### _3.2. Téléchargement automatisé du corpus_

**Objectif :** généraliser la récupération aux sept sources
du registre documentaire.

**Méthode :**
1. Parcourir les sources du registre.
2. Réutiliser les fichiers locaux existants.
3. Vérifier les empreintes SHA-256 connues.
4. Télécharger les documents manquants.
5. Contrôler les réponses HTTP et les contenus HTML.
6. Sauvegarder les documents et leurs métadonnées.

**Résultat attendu :**
- 7 documents HTML locaux ;
- 7 sources marquées `downloaded` ;
- aucune perte des fichiers déjà téléchargés.

La cellule doit pouvoir être relancée sans créer de doublons.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from urllib.parse import urlparse

import requests

# 1. Registre documentaire
with registry_path.open("r", encoding='utf-8') as file:
    registry = json.load(file)

total_sources = len(registry["sources"])

downloaded_count = 0
reused_count = 0

print(f"[INFO] Sources à vérifier : {total_sources}")


# 2. Parcours de toutes les sources du registre
for index, source in enumerate(registry["sources"], start=1):
    source_id = source["source_id"]
    source_url = source["source_url"]

    raw_path = raw_dir / f"{source_id}.html"

    print(f"\n[{index}/{total_sources}] {source['title']}")

    # 3. Vérif de la présence d'une copie locale
    if raw_path.exists():
        document = raw_path.read_bytes()

        # Vérif de l'empreinte
        checksum = hashlib.sha256(document).hexdigest()

        previous_checksum = source.get("sha_256")

        if previous_checksum is not None:
            assert checksum == previous_checksum, f"Fichier modifié : {source_id}"

        reused_count += 1
        print("[INFO] Copie locale réutilisée.")

    else:
        # 4. Téléchrgement du doc manquant
        try:
            response = requests.get(
                source_url,
                headers={
                    "User-Agent": "IndustrialVisionSLMBenchmark/1.0 (educational project)"
                }, timeout=20
            )

            response.raise_for_status()

        except requests.RequestException as error:
            raise RuntimeError(f"Echec du téléchargement : {source_id}") from error

        # 5. Vérif du type de contenu
        content_type = response.headers.get("Content-Type", "").lower()
        assert "text/html" in content_type, f"Contenu non HTML : {source_id}"

        # Vérif de l'adresse finale après redirection
        final_url = urlparse(response.url)
        assert (
            final_url.scheme == "https"
            and final_url.hostname == "docs.opencv.org"
            and final_url.path.startswith("/4.13.0/")
        ), f"URL finale inattendue : {response.url}"

        document = response.content
        print("[OK] HTTP :", response.status_code)

    # 6. Vérifier le contenu, y compris les fichiers locaux.
    assert len(document) > 1000, f"Document trop court : {source_id}"
    assert b"<html" in document[:2000].lower(), f"Structure HTML absente : {source_id}"
    assert b"opencv" in document.lower(), f"Mention OpenCV absente : {source_id}"

    # 7. Empreinte.
    checksum = hashlib.sha256(document).hexdigest()

    # Enregistrer uniquement les nouveaux documents.
    if not raw_path.exists():
        raw_path.write_bytes(document)
        source["downloaded_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
        downloaded_count += 1

        print("[OK] Document sauvegardé.")

    # 8. Mise à jour du registre.
    source["status"] = "downloaded"

    source["raw_path"] = raw_path.relative_to(project_dir).as_posix()
    source["sha256"] = checksum

    # Sauvegarder après chaque source vérifiée.
    with registry_path.open("w", encoding="utf-8") as file:
        json.dump(registry, file, ensure_ascii=False, indent=2)

    print("[OK] SHA-256 :", checksum)
    print("[OK] Source enregistrée.")


# 9. Vérifier le résultat global.
completed_sources = sum(source["status"] == "downloaded" for source in registry["sources"])

assert completed_sources == total_sources

print("\n=== Bilan du téléchargement ===")

print("Sources enregistrées :", total_sources)
print("Nouveaux téléchargements :", downloaded_count)
print("Fichiers réutilisés :", reused_count)
print("Documents disponibles :", completed_sources)

print("[OK] Corpus brut complet.")




## **4. Extraction et nettoyage des documents HTML**

### **Objectif**

Transformer les pages HTML téléchargées en documents
textuels propres, structurés et exploitables par un
système de recherche documentaire.

### **Problématique**

Les fichiers HTML contiennent à la fois des connaissances
techniques et des éléments inutiles pour notre RAG.

Nous devons isoler le contenu documentaire sans perdre
les informations importantes.

### **Méthodologie**

1. **Section 4.1 — Inspection de la structure HTML**
   - Charger une page locale.
   - Identifier ses principales balises.
   - Localiser le contenu documentaire.
   - Examiner un extrait du texte disponible.

2. **Section 4.2 — Extraction et nettoyage**
   - Isoler les éléments utiles.
   - Retirer les contenus parasites.
   - Préserver les titres et exemples techniques.
   - Enregistrer le texte nettoyé.

3. **Section 4.3 — Généralisation au corpus**
   - Traiter les sept documents.
   - Contrôler les résultats.
   - Mettre à jour les métadonnées.

### **Résultat attendu**

Sept documents textuels nettoyés et traçables,
prêts pour le découpage en chunks.

**Important :** les fichiers HTML bruts seront conservés
sans modification.

### *4.1. Inspection de la structure HTML*

#### **Objectif**

Comprendre l'organisation d'une page de documentation
OpenCV avant de développer une méthode d'extraction.

#### **Document étudié**

- Titre : Smoothing Images
- Identifiant : `opencv_filtering`
- Fichier : `data/corpus/raw/opencv_filtering.html`

#### **Outil utilisé**

Beautiful Soup (`bs4`), bibliothèque Python permettant
l'analyse et la navigation dans les documents HTML.

#### **Points à examiner**

- Titre de la page.
- Présence de titres `h1`, `h2` et `h3`.
- Présence de conteneurs HTML tels que `main`,
  `article` ou `div.textblock`.
- Quantité de texte disponible.
- Présence éventuelle de menus et autres éléments parasites.

#### **Résultat attendu**

Identifier le conteneur HTML qui représente le mieux
le contenu technique de la documentation.

Aucune modification des documents n'est effectuée
durant cette inspection.

In [ ]:
from bs4 import BeautifulSoup

# 1. Sélection du premier doc téléchargé
html_path = raw_dir / "opencv_filtering.html"
assert html_path.exists(), "Document HTML introuvable"

print("[OK] Document local trouvé :", html_path.name)

# 2. Chargement des octets du doc
html_content = html_path.read_bytes()
print("[INFO] Taille HTML :", len(html_content), " octets")

# 3. Analyse de la structure HTML
soup = BeautifulSoup(html_content, "html.parser")
assert soup.html is not None, "Structure HTML principale introuvable."

print("[OK Structure HTML analysée")

# 4. Extraction du titre de la page
page_title = soup.title.get_text(" ", strip=True) if soup.title is not None else "Titre introuvable"
print("\n=== Titre de la page ===")
print(page_title)

# 5. Recherche des principaux titres du document
headings = soup.find_all(["h1", "h2", "h3"])

print("\n=== Premiers titres HTML ===")
for heading in headings[:10]:
    print(f"{heading.name.upper()} : {heading.get_text(' ', strip=True)}")

print("[INFO] Nombre total de titres :", len(headings))

# 6. Examiner les conteneurs possibles du contenu.
candidate_selectors = ["main", "article", "div.contents", "div.textblock", "#doc-content"]
print("\n=== Conteneurs détectés ===")

for selector in candidate_selectors:
    element = soup.select_one(selector)
    if element is None:
        print(f"{selector} : absent")
        continue

    extracted_text = element.get_text(" ", strip=True)
    print(f"{selector} : {len(extracted_text)} caractères")

# 7. Afficher un extrait pour inspection manuelle.
content_element = (
    soup.select_one("div.textblock")
    or soup.select_one("div.contents")
    or soup.select_one("main")
    or soup.select_one("article")
    or soup.body
)

assert content_element is not None, ("Aucun conteneur exploitable détecté.")

preview_text = content_element.get_text(" ", strip=True)

print("\n=== Aperçu du contenu ===")
print(preview_text[:1000])

print("\n[OK] Inspection HTML terminée.")



### *4.2. Extraction structurée et nettoyage du premier document*
#### **Objectif**

Extraire le contenu technique de `opencv_filtering.html`
en conservant les informations nécessaires au futur RAG.

#### **Méthode**

1. Vérifier l'intégrité du document HTML brut.
2. Sélectionner le conteneur `div.textblock`.
3. Retirer les balises inutiles.
4. Préserver les blocs de code et leurs sauts de ligne.
5. Convertir les titres HTML en titres Markdown.
6. Conserver les paragraphes, les listes et les formules textuelles.
7. Enregistrer le document nettoyé dans `processed/`.
8. Calculer son empreinte SHA-256.

#### **Points de vigilance**

- Ne jamais modifier le document HTML brut.
- Ne pas supprimer les noms des fonctions OpenCV.
- Ne pas transformer le code Python en texte continu.
- Vérifier manuellement la conservation des formules mathématiques.
- Ne pas écraser silencieusement un fichier traité existant.

#### **Résultat attendu**

Un fichier `opencv_filtering.md` contenant la documentation
technique restructurée, avec ses titres et exemples de code.

In [ ]:
import hashlib
import json
import re
from datetime import datetime, timezone

from bs4 import BeautifulSoup, NavigableString


# 1. Chargement du registre et sélection de notre document.
with registry_path.open("r", encoding="utf-8") as file:
    registry = json.load(file)

source = next(item for item in registry["sources"] if item["source_id"] == "opencv_filtering")

html_path = raw_dir / "opencv_filtering.html"
assert html_path.is_file(), "Fichier HTML introuvable."

html_content = html_path.read_bytes()


# 2. Vérif de l'intégrité du doc.
raw_checksum = hashlib.sha256(html_content).hexdigest()
assert raw_checksum == source["sha256"], (
    "L'empreinte du document brut ne correspond pas au registre."
)

print("[OK] Intégrité du document brut vérifiée.")


# 3. Analyse du HTML et sélection de son contenu technique.
soup = BeautifulSoup(html_content,"html.parser")

content_element = soup.select_one("div.textblock")
assert content_element is not None, "Conteneur div.textblock introuvable."

# Travailler sur une copie indépendante du conteneur.
content_soup = BeautifulSoup(str(content_element), "html.parser")

content = content_soup.select_one("div.textblock")
assert content is not None


# 4. Retrait des balises non documentaires.
for element in content.select("script, style, noscript"):
    element.decompose()


# 5. Extraction des blocs de code avant de nettoyer le texte.
# Nous les remplaçons temporairement par des identifiants
# afin de préserver leur mise en forme et leur indentation.

code_placeholders = {}

code_elements = list(content.select("div.fragment, pre"))

for index, element in enumerate(code_elements):

    # Éviter de traiter deux fois un bloc <pre> qui serait
    # déjà contenu dans un fragment de code.
    if (element.name == "pre" and element.find_parent("div", class_="fragment")):
        continue

    if element.name == "div":

        # Les exemples Doxygen utilisent souvent
        # des éléments div.line pour chaque ligne de code.
        code_lines = element.select("div.line")

        if code_lines:
            code_text = "\n".join(
                line.get_text("", strip=False).replace("\xa0", " ")
                for line in code_lines
            )
        else:
            code_text = element.get_text("\n", strip=True)

    else:
        code_text = element.get_text("", strip=False)

    code_text = code_text.strip("\n")

    placeholder = f"__CODE_BLOCK_{index:03d}__"

    code_placeholders[placeholder] = f"```python\n{code_text}\n```"

    element.replace_with(NavigableString(f"\n\n{placeholder}\n\n"))

print(
    "[INFO] Blocs de code détectés :",
    len(code_placeholders)
)


# 6. Ajout de séparateurs Markdown aux titres.
for heading in content.find_all(re.compile(r"^h[1-6]$")):

    heading_level = int(heading.name[1])

    # Récupérer le texte du titre avant de supprimer sa balise.
    heading_text = heading.get_text(" ", strip=True)

    if not heading_text:
        continue

    # Décaler les titres d'un niveau :
    # h1 devient ##, h2 devient ###, etc.
    markdown_prefix = "#" * min(heading_level + 1, 6)

    # Remplacer toute la balise HTML par un titre Markdown.
    heading.replace_with(NavigableString(
            f"\n\n{markdown_prefix} {heading_text}\n\n"
        )
    )


# 7. Préservation de la séparation des paragraphes.
for paragraph in content.find_all("p"):
    paragraph.insert_after(NavigableString("\n\n"))


# 8. Préservation des sauts de ligne explicites.
for line_break in content.find_all("br"):
    line_break.replace_with(NavigableString("\n"))


# 9. Identification des éléments de liste.
for list_item in content.find_all("li"):
    list_item.insert_before(NavigableString("\n- "))
    list_item.insert_after(NavigableString("\n"))


# 10. Conversion du contenu HTML en texte.
cleaned_text = content.get_text(separator="", strip=False)
cleaned_text = cleaned_text.replace("\xa0", " ")

# Retirer les espaces inutiles en fin de ligne.
cleaned_text = re.sub(r"[ \t]+\n", "\n", cleaned_text)

# Réduire les successions de lignes vides.
cleaned_text = re.sub(r"\n[ \t]*\n(?:[ \t]*\n)+", "\n\n", cleaned_text)

# Remplacer les références isolées aux illustrations.
# Les blocs Python sont encore protégés à ce stade.
image_pattern = re.compile(r"(?mi)^[ \t]*image[ \t]*$")

cleaned_text = image_pattern.sub(
    "[Illustration non incluse dans le corpus textuel]",
    cleaned_text
)

# 11. Restauration des blocs de code.
for placeholder, code_text in code_placeholders.items():
    cleaned_text = cleaned_text.replace(placeholder, code_text)


# 12. Ajout du titre du document.
cleaned_text = f"# {source['title']}\n\n" + cleaned_text.strip() + "\n"


# 13. Contrôles de qualité.
expected_terms = ["Gaussian Blurring", "Median Blurring", "Bilateral Filtering", "cv.filter2D"]
assert len(cleaned_text) > 1500, "Texte extrait anormalement court."

for term in expected_terms:
    assert term in cleaned_text, f"Notion attendue absente : {term}"

assert "__CODE_BLOCK_" not in cleaned_text, "Un identifiant temporaire n'a pas été remplacé."

# Vérifier la bonne conversion des titres HTML.
assert "## Goals" in cleaned_text

assert "## 2D Convolution ( Image Filtering )" in cleaned_text

# Aucun titre Markdown ne doit être vide.
assert not re.search(r"(?m)^#{2,6}\s*$", cleaned_text)

print("[OK] Titres Markdown correctement structurés.")

print("[OK] Contrôles élémentaires du texte validés.")


# 14. Sauvegarde du document traité.
processed_path = processed_dir / "opencv_filtering.md"

# Encoder explicitement en UTF-8.
processed_bytes = cleaned_text.encode("utf-8")

if processed_path.exists():

    existing_text = processed_path.read_text(encoding="utf-8")

    previous_checksum = source.get("processed_sha256")

    assert previous_checksum is not None, "Empreinte précédente absente du registre."

    # Contrôler que le fichier existant correspond à la
    # version précédente enregistrée dans le registre.
    existing_checksum = hashlib.sha256(existing_text.encode("utf-8")).hexdigest()
    assert existing_checksum == previous_checksum, "Le fichier traité a été modifié hors du pipeline."

    if existing_text == cleaned_text:
        print("[INFO] Document traité déjà à jour.")

    else:
        # La modification provient de notre nouvel extracteur.
        # Écriture binaire pour préserver exactement les octets UTF-8.
        processed_path.write_bytes(processed_bytes)
        print("[OK] Document traité régénéré.")

else:
    processed_path.write_bytes(processed_bytes)
    print("[OK] Document traité sauvegardé.")



# 15. Calculer l'empreinte du document nettoyé.
processed_checksum = hashlib.sha256(cleaned_text.encode("utf-8")).hexdigest()

source["processed_path"] = processed_path.relative_to(project_dir).as_posix()
source["processed_sha256"] = processed_checksum

source.setdefault("processed_at", datetime.now(timezone.utc).isoformat(timespec="seconds"))


# 16. Sauvegarde des nouvelles métadonnées.
with registry_path.open("w", encoding="utf-8") as file:
    json.dump(registry, file, ensure_ascii=False, indent=2)

print("[OK] Registre mis à jour.")


# 17. Affichage du bilan de l'extraction.
print("\n=== Bilan de l'extraction ===")
print("Document :", source["title"])
print("Fichier :", source["processed_path"])
print("Caractères :", len(cleaned_text))
print("SHA-256 :", processed_checksum)

print("\n=== Aperçu du texte extrait ===")
print(cleaned_text[:1800])

print("\n[OK] Section 4.2 exécutée.")


### *4.3. Généralisation de l'extraction au corpus*

#### **Objectif**

Étendre notre méthode d'extraction structurée aux sept
documents OpenCV du registre.

#### **Problématique**

La structure HTML observée sur `Smoothing Images`
n'est pas nécessairement identique sur les autres pages.

Avant de généraliser notre extracteur, nous devons
vérifier que les documents utilisent des conteneurs
compatibles.

#### **Méthodologie**

1. Contrôler l'intégrité des sept fichiers HTML.
2. Vérifier la présence de `div.textblock`.
3. Compter les titres et les blocs de code.
4. Identifier les éventuelles différences de structure.
5. Généraliser l'extracteur en fonction des observations.
6. Produire les sept documents Markdown.
7. Vérifier et enregistrer les résultats.

#### **Résultat attendu**

Sept documents structurés dans `data/corpus/processed/`,
avec des métadonnées et des empreintes SHA-256.

Les fichiers HTML bruts devront rester inchangés.

In [ ]:
import hashlib
import json

from bs4 import BeautifulSoup

# 1. Registre documentaire
with registry_path.open("r", encoding="utf-8") as file:
    registry = json.load(file)

print("[INFO] Documents à inspecter :", len(registry["sources"]))

# 2. Préparation des résultats de l'audit.
audit_results = []
# 3. Parcourir tous les documents HTML.
for source in registry["sources"]:

    source_id = source["source_id"]

    html_path = project_dir / source["raw_path"]

    assert html_path.is_file(), (
        f"Fichier HTML introuvable : {source_id}"
    )

    # Lire le document sans le modifier.
    html_content = html_path.read_bytes()

    # Contrôler son intégrité.
    checksum = hashlib.sha256(
        html_content
    ).hexdigest()

    assert checksum == source["sha256"], (
        f"Empreinte incorrecte : {source_id}"
    )

    # Analyser les balises HTML.
    soup = BeautifulSoup(
        html_content,
        "html.parser"
    )

    content_element = soup.select_one(
        "div.textblock"
    )

    # Déterminer la présence du conteneur principal.
    has_textblock = content_element is not None

    # Compter les éléments documentaires.
    if has_textblock:

        headings_count = len(
            content_element.find_all(
                ["h1", "h2", "h3"]
            )
        )

        fragments_count = len(content_element.select("div.fragment"))
        text_length = len(content_element.get_text(" ", strip=True))

    else:
        headings_count = 0
        fragments_count = 0
        text_length = 0

    audit_results.append({
        "source_id": source_id,
        "has_textblock": has_textblock,
        "headings_count": headings_count,
        "fragments_count": fragments_count,
        "text_length": text_length
    })

# 4. Afficher un résumé lisible.
print("\n=== Audit des documents HTML ===")

for result in audit_results:
    print(f"\nDocument : {result['source_id']}")

    print("Conteneur textblock :", "OUI" if result["has_textblock"] else "NON")

    print("Titres h1-h3 :", result["headings_count"])
    print("Fragments de code :", result["fragments_count"])
    print("Longueur du texte :", result["text_length"], "caractères")


# 5. Calculer le bilan.
compatible_count = sum( result["has_textblock"] for result in audit_results)

print("\n=== Bilan de compatibilité ===")

print("Documents inspectés :", len(audit_results))
print("Documents compatibles :", compatible_count)
print("Documents sans textblock :", len(audit_results) - compatible_count)

print("[OK] Audit HTML terminé.")


### **Extraction automatisée des 7 docs**

**Objectif:** generaliser l'extracteur validé dans la section 4.2 à l'ensemble du corpus documentaire

**Méthode :**

1. Transformer la logique d'extraction en fonction réutilisable.
2. Parcourir les sept documents HTML.
3. Vérifier l'intégrité de chaque document brut.
4. Extraire les titres, paragraphes et blocs de code.
5. Contrôler la structure Markdown obtenue.
6. Sauvegarder les fichiers dans `processed/`.
7. Enregistrer les empreintes SHA-256 dans le registre.

**Contraintes :**

- Ne jamais modifier les documents HTML bruts.
- Ne pas écraser un fichier traité modifié manuellement.
- Permettre la réexécution sans modification inutile.
- Conserver la correspondance entre chaque fichier
  et sa source officielle.

**Résultat attendu :**

Sept fichiers Markdown structurés, chacun associé
à une source du registre documentaire.

Le nettoyage des références aux illustrations et
le contrôle qualité approfondi seront réalisés ensuite.

In [ ]:
import hashlib
import json
import re
from datetime import datetime, timezone

from bs4 import BeautifulSoup, NavigableString


# 1. Définir une fonction d'extraction réutilisable.
def extract_markdown(html_content, document_title):

    # Analyser la structure HTML.
    soup = BeautifulSoup(html_content, "html.parser")

    content_element = soup.select_one("div.textblock")
    assert content_element is not None, "Conteneur div.textblock introuvable."

    # Travailler sur une copie du contenu documentaire.
    content_soup = BeautifulSoup(str(content_element), "html.parser")

    content = content_soup.select_one("div.textblock")
    assert content is not None

    # Supprimer les éléments non documentaires.
    for element in content.select("script, style, noscript"):
        element.decompose()

    # Protéger temporairement les blocs de code.
    code_placeholders = {}

    code_elements = list(content.select("div.fragment, pre"))

    for index, element in enumerate(code_elements):
        # Éviter le double traitement d'un bloc imbriqué.
        if (element.name == "pre" and element.find_parent("div", class_="fragment")):
            continue

        if element.name == "div":
            code_lines = element.select("div.line")
            if code_lines:
                code_text = "\n".join(
                    line.get_text("", strip=False).replace("\xa0", " ")
                    for line in code_lines
                )

            else:
                code_text = element.get_text("\n", strip=True)

        else:
            code_text = element.get_text("", strip=False)

        code_text = code_text.strip("\n")

        placeholder = f"__CODE_BLOCK_{index:03d}__"

        code_placeholders[placeholder] = f"```python\n{code_text}\n```"

        element.replace_with(NavigableString(f"\n\n{placeholder}\n\n"))

    # Convertir les titres HTML en titres Markdown.
    for heading in content.find_all(re.compile(r"^h[1-6]$")):
        heading_level = int(heading.name[1])

        heading_text = heading.get_text(" ", strip=True)

        if not heading_text:
            continue

        markdown_prefix = "#" * min(heading_level + 1, 6)

        heading.replace_with(NavigableString(f"\n\n{markdown_prefix} {heading_text}\n\n"))

    # Séparer les paragraphes.
    for paragraph in content.find_all("p"):
        paragraph.insert_after(NavigableString("\n\n"))

    # Préserver les sauts de ligne explicites.
    for line_break in content.find_all("br"):
        line_break.replace_with(NavigableString("\n"))

    # Préserver les éléments de liste.
    for list_item in content.find_all("li"):
        list_item.insert_before(NavigableString("\n- "))
        list_item.insert_after(NavigableString("\n"))

    # Extraire le texte.
    cleaned_text = content.get_text(separator="", strip=False)

    # Normaliser les espaces.
    cleaned_text = cleaned_text.replace("\xa0", " ")
    cleaned_text = re.sub(r"[ \t]+\n", "\n", cleaned_text)
    cleaned_text = re.sub(r"\n[ \t]*\n(?:[ \t]*\n)+", "\n\n", cleaned_text)


    # Remplacer les mentions isolées aux illustrations.
    # Les exemples de code sont encore protégés par leurs
    # identifiants temporaires : ils ne seront pas modifiés.

    image_pattern = re.compile(r"(?mi)^[ \t]*image[ \t]*$")
    cleaned_text = image_pattern.sub("[Illustration non incluse dans le corpus textuel]", cleaned_text)

    # Restaurer les blocs de code.
    for placeholder, code_text in (code_placeholders.items()):
        cleaned_text = cleaned_text.replace(placeholder, code_text)

    # Ajouter le titre principal du document.
    cleaned_text = f"# {document_title}\n\n" + cleaned_text.strip() + "\n"

    # Vérifications élémentaires.
    assert len(cleaned_text) > 1000, "Document extrait anormalement court."
    assert "__CODE_BLOCK_" not in cleaned_text, "Identifiant temporaire non remplacé."
    assert not re.search(r"(?m)^#{2,6}\s*$", cleaned_text), "Titre Markdown vide détecté."

    # Vérifier l'absence des anciennes mentions isolées.
    assert not re.search(r"(?mi)^[ \t]*image[ \t]*$", cleaned_text), "Référence isolée à une image encore présente."
 
    return cleaned_text, len(code_placeholders)


# 2. Charger le registre des sources.
with registry_path.open("r", encoding="utf-8") as file:
    registry = json.load(file)

created_count = 0
updated_count = 0
reused_count = 0

print("[INFO] Documents à traiter :", len(registry["sources"]))


# 3. Parcourir les sept sources documentaires.
for source in registry["sources"]:
    source_id = source["source_id"]

    print(f"\n=== {source_id} ===")

    html_path = project_dir / source["raw_path"]

    assert html_path.is_file(), f"Document HTML absent : {source_id}"
    

    html_content = html_path.read_bytes()

    # Vérifier l'intégrité du document original.
    raw_checksum = hashlib.sha256(html_content).hexdigest()

    assert raw_checksum == source["sha256"], f"Intégrité HTML incorrecte : {source_id}"

    # Extraire le contenu technique.
    cleaned_text, code_count = extract_markdown(html_content, source["title"])

    processed_bytes = cleaned_text.encode("utf-8")

    processed_checksum = hashlib.sha256(processed_bytes).hexdigest()

    processed_path = processed_dir / f"{source_id}.md"

    # 4. Contrôler les fichiers déjà présents.
    if processed_path.exists():
        existing_text = processed_path.read_text(encoding="utf-8")

        existing_checksum = hashlib.sha256(existing_text.encode("utf-8")).hexdigest()
        previous_checksum = source.get("processed_sha256")

        assert previous_checksum is not None, f"Empreinte précédente absente : {source_id}"

        assert existing_checksum == previous_checksum, f"Fichier modifié hors du pipeline : {source_id}"
        
        if existing_text == cleaned_text:
            reused_count += 1
            print("[INFO] Document déjà à jour.")

        else:
            # Régénération autorisée après vérification
            # de l'intégrité de la version précédente.
            processed_path.write_bytes(processed_bytes)

            updated_count += 1

            print("[OK] Document régénéré.")

    else:
        # Création du document traité.
        processed_path.write_bytes(processed_bytes)
        created_count += 1

        print("[OK] Document créé.")


    # 5. Enregistrer les métadonnées.

    # Comparer avec l'empreinte précédente avant de la remplacer.
    file_changed = source.get("processed_sha256") != processed_checksum

    source["processed_path"] = processed_path.relative_to(project_dir).as_posix()
    source["processed_sha256"] = processed_checksum

    # Actualiser la date uniquement si le document a changé
    # ou si aucune date n'avait encore été enregistrée.
    if file_changed or not source.get("processed_at"):
        source["processed_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")


    # Sauvegarder le registre après chaque document.
    with registry_path.open("w", encoding="utf-8") as file:
        json.dump(registry, file, ensure_ascii=False, indent=2)

    print("[OK] Caractères :", len(cleaned_text))
    print("[OK] Blocs de code :", code_count)

    print("[OK] SHA-256 :", processed_checksum)


# 6. Afficher le bilan final.
print("\n=== Bilan du corpus traité ===")

print("Documents créés :", created_count)
print("Documents régénérés :", updated_count)
print("Documents réutilisés :", reused_count)

processed_count = sum("processed_sha256" in source for source in registry["sources"])

print("Documents traités :", processed_count)

assert processed_count == len(registry["sources"])

print("[OK] Extraction du corpus terminée.")

### *4.4. Contrôle qualité du corpus documentaire*

#### **Objectif**

Vérifier l'intégrité et la structure des sept documents
Markdown avant leur découpage en chunks.

#### **Contrôles automatisés**

1. Vérifier la présence des fichiers HTML et Markdown.
2. Comparer leurs empreintes SHA-256 au registre.
3. Vérifier la présence du titre principal.
4. Détecter les titres Markdown vides.
5. Vérifier l'ouverture et la fermeture des blocs de code.
6. Rechercher les identifiants temporaires non remplacés.
7. Repérer les références isolées à des images.

#### **Contrôle manuel ultérieur**

Examiner les anomalies détectées afin de décider
si elles doivent être corrigées, conservées ou documentées.

#### **Résultat attendu**

Un bilan de qualité pour chacun des sept documents.

Aucun document n'est modifié pendant cet audit.

In [ ]:
import hashlib
import json
import re
from collections import Counter


# 1. Charger le registre documentaire.
with registry_path.open("r", encoding="utf-8") as file:
    registry = json.load(file)

quality_reports = []

print("[INFO] Documents à contrôler :", len(registry["sources"]))

# 2. Examiner les documents un par un.
for source in registry["sources"]:
    source_id = source["source_id"]

    raw_path = project_dir / source["raw_path"]
    processed_path = project_dir / source["processed_path"]

    # Vérifier la présence des deux fichiers.
    assert raw_path.is_file(), f"Document HTML manquant : {source_id}"
    assert processed_path.is_file(), f"Document Markdown manquant : {source_id}"

    # 3. Vérifier les empreintes SHA-256.
    raw_checksum = hashlib.sha256(raw_path.read_bytes()).hexdigest()
    processed_bytes = processed_path.read_bytes()

    processed_checksum = hashlib.sha256(processed_bytes).hexdigest()
    assert raw_checksum == source["sha256"], f"HTML modifié : {source_id}"
    assert processed_checksum == source["processed_sha256"], f"Markdown modifié : {source_id}"

    # 4. Lire le document Markdown.
    document_text = processed_bytes.decode("utf-8")

    # Vérifier le titre principal.
    expected_title = f"# {source['title']}"
    assert document_text.startswith(expected_title + "\n"), f"Titre principal incorrect : {source_id}"

    # 5. Examiner les titres Markdown.
    headings = re.findall(r"(?m)^#{1,6} [^\n]+$", document_text)
    
    empty_headings = re.findall(r"(?m)^#{1,6}[ \t]*$", document_text)
    assert not empty_headings, f"Titres vides détectés : {source_id}"

    # 6. Vérifier les délimiteurs des blocs de code.
    code_fences = re.findall(r"(?m)^```[^\n]*$", document_text)

    assert len(code_fences) % 2 == 0, f"Bloc de code non fermé : {source_id}"

    code_count = len(code_fences) // 2

    # 7. Vérifier les identifiants temporaires.
    assert not re.search(r"__CODE_BLOCK_\d+__", document_text), f"Identifiant temporaire détecté : {source_id}"

    # 8. Repérer les références isolées aux images.
    # Une ligne contenant uniquement "image" peut signaler
    # une illustration qui n'a pas été conservée.
    image_references = []

    for line_number, line in enumerate(document_text.splitlines(), start=1):
        if re.fullmatch(r"\s*image\s*", line, flags=re.IGNORECASE):
            image_references.append(line_number)

    # 9. Enregistrer les observations.
    quality_reports.append({
        "source_id": source_id,
        "topic": source["topic"],
        "characters": len(document_text),
        "headings": len(headings),
        "code_blocks": code_count,
        "image_references": image_references,
        "sha256_valid": True
    })

# 10. Afficher le rapport de contrôle.
print("\n=== Audit qualité du corpus ===")

for report in quality_reports:

    print(f"\nDocument : {report['source_id']}")
    print("Caractères :", report["characters"])
    print("Titres Markdown :", report["headings"])
    print("Blocs de code :", report["code_blocks"])

    print("Références isolées aux images :", len(report["image_references"]))

    if report["image_references"]:
        print("[À EXAMINER] Lignes :", report["image_references"][:10])

    print("[OK] Intégrité et structure vérifiées.")


# 11. Calculer le bilan global.
total_characters = sum(report["characters"] for report in quality_reports)
total_code_blocks = sum(report["code_blocks"] for report in quality_reports)
total_image_references = sum(len(report["image_references"]) for report in quality_reports)

topic_counts = Counter(report["topic"]for report in quality_reports)

assert len(quality_reports) == 7
assert len(topic_counts) == 5


print("\n=== Bilan global ===")

print("Documents contrôlés :", len(quality_reports))
print("Domaines représentés :", len(topic_counts))
print("Caractères extraits :", total_characters)
print("Blocs de code :", total_code_blocks)
print("Références aux images à examiner :", total_image_references)

print("[OK] Audit structurel terminé.")

### *4.5. Analyse des références aux illustrations*

#### **Objectif**

Examiner les 32 références isolées aux images détectées
lors du contrôle qualité du corpus.

#### **Problématique**

Le texte extrait contient des mentions `image` qui ne
représentent pas nécessairement une information technique
exploitable par notre futur système RAG.

#### **Méthode**

1. Repérer les lignes contenant uniquement `image`.
2. Afficher leur contexte textuel.
3. Examiner les balises `<img>` des documents HTML.
4. Déterminer si les illustrations apportent une
   information technique importante.
5. Choisir une stratégie de nettoyage adaptée.

#### **Résultat attendu**

Identifier les références parasites et les informations
visuelles manquantes avant de finaliser le corpus.

**Important :** cette étape est un diagnostic.
Aucun document brut ou traité ne sera modifié.

In [ ]:
import json
import re

from bs4 import BeautifulSoup


# 1. Recharger le registre documentaire.
with registry_path.open("r", encoding="utf-8") as file:
    registry = json.load(file)

total_references = 0

# 2. Parcourir les sept documents.
for source in registry["sources"]:
    source_id = source["source_id"]

    processed_path = project_dir / source["processed_path"]
    raw_path = project_dir / source["raw_path"]

    # Lire les lignes du Markdown.
    document_lines = processed_path.read_text(encoding="utf-8").splitlines()

    # Repérer les lignes contenant uniquement "image".
    image_indices = [
        index for index, line in enumerate(document_lines)
        if re.fullmatch(r"\s*image\s*", line, flags=re.IGNORECASE)
    ]

    total_references += len(image_indices)

    # 3. Compter les images présentes dans le HTML brut.
    soup = BeautifulSoup(raw_path.read_bytes(), "html.parser")

    content_element = soup.select_one("div.textblock")
    assert content_element is not None

    image_elements = content_element.find_all("img")

    print(f"\n=== {source_id} ===")
    print("Références Markdown :", len(image_indices))
    print("Balises HTML img :", len(image_elements))

    # 4. Examiner le contexte des trois premières références.
    for index in image_indices[:3]:
        print(f"\n[IMAGE] Ligne {index + 1}")
        start_index = max(0, index - 3)
        end_index = min(len(document_lines), index + 4)

        for line_index in range(start_index, end_index):
            line_text = document_lines[line_index].strip()
            if line_text:
                print(f"  {line_index + 1}: {line_text[:160]}")

    # 5. Examiner les métadonnées de deux images HTML.
    for image_index, image_element in enumerate(image_elements[:2], start=1):
        print(f"\n[HTML IMAGE {image_index}]")
        print("Alt :", image_element.get("alt", ""))
        print("Source :", image_element.get("src", ""))


# 6. Bilan global.
print("\n=== Bilan des références ===")

print("Documents inspectés :", len(registry["sources"]))
print("Références Markdown détectées :", total_references)

print("[OK] Inspection des illustrations terminée.")

### *4.6. Traitement des références aux illustrations*

#### **Objectif**

Éliminer les mentions isolées `image` qui représentent
des illustrations non conservées dans le corpus textuel.

#### **Constat**

L'audit a identifié 32 références isolées aux images
dans les sept documents Markdown.

Ces références sont insuffisantes pour décrire les
informations visuelles des documents d'origine.

#### **Stratégie retenue**

Remplacer chaque mention isolée `image` par :

`[Illustration non incluse dans le corpus textuel]`

Cette indication permet de signaler l'absence de
l'illustration sans inventer son contenu.

#### **Contraintes**

- Conserver les documents HTML originaux.
- Ne pas modifier les exemples de code Python.
- Préserver les titres et les formules mathématiques.
- Régénérer les documents avec leurs nouvelles empreintes.
- Vérifier que les références parasites ont disparu.

#### **Résultat attendu**

Sept documents Markdown régénérés et vérifiés,
sans mention isolée `image`.

Cette transformation ne constitue pas une extraction
multimodale : les illustrations restent absentes.


## **5. Découpage du corpus en chunks**

### **Objectif**

Transformer les sept documents Markdown en passages
documentaires adaptés à la recherche sémantique du RAG.

### **Problématique**

Un document complet peut contenir plusieurs sujets distincts.

Nous devons permettre au moteur de recherche de retrouver
uniquement les parties utiles à une question.

### **Stratégie envisagée**

Utiliser un découpage hybride :

1. Identifier les sections à partir des titres Markdown.
2. Conserver les informations associées à chaque section.
3. Regrouper les petits passages lorsque cela est pertinent.
4. Subdiviser les sections trop longues.
5. Éviter de couper les blocs de code et les formules.
6. Conserver les métadonnées de provenance.

### **Étapes prévues**

- **5.1.** Analyse de la structure des documents.
- **5.2.** Définition de la stratégie de découpage.
- **5.3.** Implémentation du chunking.
- **5.4.** Génération et sauvegarde des chunks.
- **5.5.** Contrôle qualité des chunks.

### **Résultat attendu**

Un ensemble de chunks structurés et traçables,
prêts pour une future indexation vectorielle.

### *5.1. Analyse de la structure des documents*

#### **Objectif**

Étudier la répartition et la taille des sections Markdown
avant de définir notre stratégie de chunking.

#### **Méthode**

1. Charger les sept documents traités.
2. Identifier les titres Markdown hors des blocs de code.
3. Délimiter les sections documentaires.
4. Mesurer la longueur des sections.
5. Repérer les sections particulièrement longues.

#### **Résultat attendu**

Pour chaque document :

- Nombre de sections.
- Taille moyenne d'une section.
- Taille de la plus longue section.
- Nombre de sections dépassant 1 500 caractères.

Le seuil de 1 500 caractères est utilisé uniquement
pour ce diagnostic. Il ne constitue pas encore la
taille définitive de nos futurs chunks.

Aucun fichier n'est modifié.

In [ ]:
import json
import re
from statistics import mean


# 1. Charger le registre documentaire.
with registry_path.open("r", encoding="utf-8") as file:
    registry = json.load(file)


# 2. Définir une fonction pour identifier les sections.
def extract_sections(document_text):
    # Conserver les retours à la ligne.
    lines = document_text.splitlines(keepends=True)

    sections = []

    current_title = "Introduction"
    current_level = 0
    current_lines = []

    inside_code = False

    for line in lines:
        # Détecter les délimiteurs de code Markdown.
        if line.lstrip().startswith("```"):
            inside_code = not inside_code

        # Ne pas interpréter les titres à l'intérieur du code.
        heading_match = None

        if not inside_code:
            heading_match = re.match(r"^(#{1,6})[ \t]+(.+?)\s*$", line)

        if heading_match is not None:
            # Enregistrer la section précédente.
            if current_lines:
                section_text = "".join(current_lines)

                if section_text.strip():
                    sections.append({
                        "title": current_title,
                        "level": current_level,
                        "text": section_text,
                        "characters": len(section_text)
                    })

            # Initialiser une nouvelle section.
            current_title = heading_match.group(2)
            current_level = len(heading_match.group(1))

            current_lines = [line]

        else:
            current_lines.append(line)

    # Enregistrer la dernière section.
    if current_lines:
        section_text = "".join(current_lines)

        if section_text.strip():
            sections.append({
                "title": current_title,
                "level": current_level,
                "text": section_text,
                "characters": len(section_text)
            })

    # Un bloc de code ouvert doit être refermé.
    assert not inside_code, "Bloc de code non fermé."

    return sections


# 3. Examiner les sept documents.
section_reports = []

for source in registry["sources"]:

    processed_path = project_dir / source["processed_path"]

    document_text = processed_path.read_text(encoding="utf-8")
    sections = extract_sections(document_text)

    section_lengths = [section["characters"]for section in sections]
    assert section_lengths, f"Aucune section trouvée : {source['source_id']}"

    longest_section = max(sections, key=lambda section: section["characters"])

    long_sections = [section for section in sections if section["characters"] > 1500]

    report = {
        "source_id": source["source_id"],
        "sections": len(sections),
        "average_length": round(mean(section_lengths), 1),
        "max_length": longest_section["characters"],
        "longest_title": longest_section["title"],
        "long_sections": len(long_sections)
    }

    section_reports.append(report)

    print(f"\n=== {source['source_id']} ===")
    print("Nombre de sections :", report["sections"])
    print("Longueur moyenne :", report["average_length"], "caractères")
    print("Section la plus longue :", report["longest_title"])
    print("Longueur maximale :", report["max_length"], "caractères")
    print("Sections > 1500 caractères :", report["long_sections"])


# 4. Afficher le bilan global.
total_sections = sum(report["sections"] for report in section_reports)
total_long_sections = sum(report["long_sections"] for report in section_reports)

print("\n=== Bilan de structure ===")

print("Documents analysés :", len(section_reports))
print("Sections identifiées :", total_sections)

print("Sections > 1500 caractères :", total_long_sections)

print("[OK] Analyse des sections terminée.")

### *5.2. Définition de la stratégie de découpage*

#### **Objectif**

Définir une stratégie de chunking hybride qui respecte
la structure des documents et les unités de contenu.

#### **Résultats de l'analyse précédente**

- 7 documents.
- 57 sections Markdown.
- 10 sections dépassant 1 500 caractères.

#### **Stratégie retenue**

1. Conserver les sections de taille raisonnable.
2. Découper les sections trop longues entre leurs blocs.
3. Préserver les blocs de code complets.
4. Préserver les formules mathématiques multilignes.
5. Signaler les blocs dépassant la taille cible.
6. Conserver la provenance de chaque futur chunk.

#### **Paramètre initial**

`target_chunk_size = 1500`

Cette valeur représente un objectif exprimé en caractères,
et non une limite stricte en tokens.

#### **Expérience**

Analyser la longueur des blocs élémentaires pour déterminer
si notre stratégie peut respecter cette taille cible.

#### **Résultat attendu**

- Nombre total de blocs élémentaires.
- Taille du plus grand bloc.
- Nombre de blocs dépassant 1 500 caractères.
- Identification des éventuelles exceptions.

Aucun fichier du corpus ne sera modifié.

In [ ]:
import json

# 1. Taille chunk
target_chunk_size = 1500

# 2. Découpage d'une section en blocs élémentaires
def extract_atomic_blocks(section_text):
    blocks = []
    current_lines = []

    inside_code = False
    math_depth = 0

    for line in section_text.splitlines(keepends=True):
        stripped_line = line.strip()

        # Détecter les délimiteurs de code Markdown
        if stripped_line.startswith("```"):
            current_lines.append(line)
            inside_code = not inside_code
            continue

        # Suivre les formules délimitées par \[ et \].
        # Ne pas interpréter le contenu des blocs Python comme des formules documentaires.
        if not inside_code:
            math_depth += line.count(r"\[")
            math_depth -= line.count(r"\]")

            assert math_depth >= 0, "Délimiteur mathématique fermé sans ouverture."

        # Une ligne vide délimite un bloc uniquement si nous ne sommes ni dans du code ni dans une formule mathématique multiligne
        if(not stripped_line and not inside_code and math_depth == 0):
            if current_lines:
                block_text = "".join(current_lines).strip()
                if block_text:
                    blocks.append(block_text)

                current_lines = []

        else:
            current_lines.append(line)

    # Vérifier que les éléments protégés sont fermés
    assert not inside_code, "Bloc de code non fermé"
    assert math_depth == 0, "Formule mathématique non fermée"

    # Conserver le dernier bloc
    if current_lines:
        block_text = "".join(current_lines).strip()
        if block_text:
            blocks.append(block_text)

    return blocks

# 3. Chargement du registre
with registry_path.open("r", encoding="utf-8") as file:
    registry = json.load(file)

# 4. Analyse des blocs de chaque doc
block_reports = []
oversized_blocks = []

for source in registry["sources"]:
    document_path = project_dir / source['processed_path']
    document_text = document_path.read_text(encoding="utf-8")

    # Réutiliser la fonction de la section 5.1
    sections = extract_sections(document_text)

    document_blocks = []
    for section in sections:
        blocks = extract_atomic_blocks(section['text'])

        for block in blocks:
            block_info = {
                "source_id": source["source_id"],
                "section": section['title'],
                "characters": len(block),
                "preview": block[:160]
            }

            document_blocks.append(block_info)

            if len(block) > target_chunk_size:
                oversized_blocks.append(block_info)
    assert document_blocks, f"Aucun bloc trouvé : {source['source_id']}"

    longest_block = max(document_blocks, key=lambda block: block['characters'])

    report = {
        "source_id": source['source_id'],
        "blocks": len(document_blocks),
        "max_length": longest_block["characters"],
        "oversized": sum(block["characters"] > target_chunk_size for block in document_blocks)
    }

    block_reports.append(report)

    print(f"\n=== {source['source_id']} ===")
    print("Blocs identifiés :", report["blocks"])
    print("Plus grand bloc :", report["max_length"])
    print("Blocs > 1500 :", report["oversized"])

# 5. Affichage du bilan global.
print("\n=== Bilan des blocs élémentaires ===")

print("Documents analysés :", len(block_reports))
print("Blocs identifiés :", sum(report["blocks"] for report in block_reports))
print("Blocs > 1500 caractères :", len(oversized_blocks))


# 6. Affichage des exceptions éventuelles.
if oversized_blocks:
    print("\n=== Blocs à examiner ===")

    for block in sorted(oversized_blocks, key=lambda item: item["characters"], reverse=True)[:5]:
        print("\nDocument :", block["source_id"])
        print("Section :", block["section"])
        print("Taille :", block["characters"])
        print("Aperçu :", block["preview"])

else:
    print("[OK] Aucun bloc élémentaire ne dépasse la taille cible.")

print("\n[OK] Analyse des blocs terminée.")


### *5.3. Implémentation du découpage en chunks*

#### **Objectif**

Construire une fonction permettant de découper les sections
Markdown en chunks de 1 500 caractères maximum.

#### **Stratégie**

1. Identifier les sections du document.
2. Extraire leurs blocs élémentaires.
3. Regrouper progressivement les blocs.
4. Commencer un nouveau chunk si la taille cible est dépassée.
5. Répéter le titre de la section dans chaque chunk.
6. Conserver les métadonnées documentaires.

#### **Contraintes**

- Ne pas diviser un bloc élémentaire.
- Ne pas mélanger deux sections différentes.
- Inclure les titres dans le calcul de la taille.
- Préserver les blocs de code complets.
- Signaler toute impossibilité de respecter la taille cible.

#### **Première expérience**

Appliquer le chunking au document `opencv_filtering`.

#### **Résultat attendu**

Une liste de dictionnaires Python contenant :

- un identifiant unique ;
- le document d'origine ;
- le domaine technique ;
- la section ;
- l'URL de la source ;
- le texte du chunk ;
- sa longueur en caractères.

Aucun fichier n'est encore sauvegardé.

In [ ]:
import hashlib
import json

# 1. Fonction de chunking
def create_document_chunks(source, document_text, max_chars=1500):
    sections = extract_sections(document_text)

    chunks = []
    assigned_blocks = 0

    # 2. Parcours de chaque section Markdown
    for section_index, section in enumerate(sections, start=1):
        section_level = section['level']
        section_title = section['title']

        # Reconstituer le titre Markdown de la section
        heading = f"{'#' * section_level} {section_title}"
        section_text = section['text']
        assert section_text.startswith(heading), f"Titre de section inattendu : {section_title}"

        # Séparer le titre et le corps de la section
        body_text = section_text[len(heading):].strip()

        # Ignorer les sections qui ne contiennent pas de texte après leur titre
        if not body_text:
            continue

        # Extraire les blocs sans les diviser
        blocks = extract_atomic_blocks(body_text)

        current_blocks = []
        part_index = 1

        # 3. Fonction interne pour enregistrer un chunk
        def save_chunk(selected_blocks, part_number):
            chunk_body = "\n\n".join(selected_blocks)
            chunk_text = f"{heading}\n\n{chunk_body}"
            assert len(chunk_text) <= max_chars, f"Chunk trop long : {section_title}"

            chunk_id = f"{source['source_id']}_chunk_{len(chunks) + 1:03d}"

            chunks.append({
                "chunk_id": chunk_id,
                "document_id": source["source_id"],
                "topic": source["topic"],
                "document_title": source['title'],
                "section": section_title,
                "section_index": section_index,
                "part_index": part_number,
                "source_url": source['source_url'],
                "text": chunk_text,
                "characters": len(chunk_text)
            })

        # 4. Regrouper progressivement les blocs
        for block in blocks:
            candidate_blocks = current_blocks + [block]
            candidate_body = "\n\n".join(candidate_blocks)
            candidate_text = f"{heading}\n\n{candidate_body}"

            if len(candidate_text) <= max_chars:
                # Le bloc tient dans le chunk courant
                current_blocks.append(block)
            else:
                # Terminer le chunk courant si nécessaire
                if current_blocks:
                    save_chunk(current_blocks, part_index)
                    part_index += 1
                    current_blocks = []

                # Vérifier que le bloc peut tenir seul, accompagné du titre de sa section
                single_block_text = f"{heading}\n\n{block}"
                if len(single_block_text) > max_chars:
                    raise ValueError(f"Bloc indivisible trop long : {source['source_id']} / {section_title} / {len(single_block_text)} caractères")

                current_blocks = [block]
            assigned_blocks += 1

        # 5. Enregistrement du dernier chunk de la section
        if current_blocks:
            save_chunk(current_blocks, part_index)
    
    return chunks, assigned_blocks

# 6. Chargement du registre documentaire
with registry_path.open('r', encoding="utf-8") as file:
    registry = json.load(file)

# 7. Sélection de notre document de test
source = next(item for item in registry["sources"] if item["source_id"] == "opencv_filtering")
document_path = project_dir / source["processed_path"]
document_bytes = document_path.read_bytes()

# Vérifier l'intégrité avant le découpage
assert hashlib.sha256(document_bytes).hexdigest() == source["processed_sha256"]

document_text = document_bytes.decode("utf-8")

# 8. Génération des chunks en mémoire
test_chunks, assigned_blocks = create_document_chunks(source, document_text, target_chunk_size)

# 9. Vérif des chunks obtenus
assert test_chunks, "Aucun chunk généré"

chunk_ids = [chunk["chunk_id"] for chunk in test_chunks]
assert len(chunk_ids) == len(set(chunk_ids)), "Identifiants de chunks dupliqués."

for chunk in test_chunks:
    assert chunk["characters"] <= target_chunk_size
    assert chunk["characters"] == len(chunk["text"])

    # Vérifier que les délimiteurs de code sont équilibrés dans chaque chunk
    assert chunk["text"].count("```") % 2 == 0, f"Bloc de code incomplet : {chunk['chunk_id']}"

# 10. Vérif de la conservation des blocs de code
original_code_blocks = document_text.count("```python")
chunk_code_blocks = sum(chunk["text"].count("```python") for chunk in test_chunks)
assert original_code_blocks == chunk_code_blocks, "Nombre de blocs Python modifié pendant le découpage."

# 11. Affichage du bilan.
print("\n=== Premier test de chunking ===")

print("Document :", source["source_id"])
print("Sections :", len(extract_sections(document_text)))
print("Blocs affectés :", assigned_blocks)
print("Chunks générés :", len(test_chunks))

print("Taille maximale :", max(chunk["characters"] for chunk in test_chunks))

print("Blocs Python conservés :", chunk_code_blocks, "/", original_code_blocks)

print("\n=== Aperçu des premiers chunks ===")

for chunk in test_chunks[:3]:
    print("\nID :", chunk["chunk_id"])
    print("Section :", chunk["section"])
    print("Taille :", chunk["characters"])

    print("Texte :")
    print(chunk["text"][:350])

    print("-" * 40)

print("\n[OK] Premier test de chunking terminé.")


### *5.4. Génération et sauvegarde des chunks*

#### **Objectif**

Appliquer la fonction de chunking validée aux sept documents
OpenCV et sauvegarder les passages obtenus en JSONL.

#### **Méthode**

1. Charger le registre documentaire.
2. Vérifier l'intégrité des fichiers Markdown.
3. Générer les chunks de chaque document.
4. Vérifier leurs tailles, identifiants et blocs de code.
5. Regrouper les chunks dans un fichier JSONL.
6. Enregistrer un manifeste de reproductibilité.

#### **Configuration**

- Stratégie : découpage hiérarchique par sections.
- Taille maximale : 1 500 caractères.
- Unités préservées : paragraphes, formules et code.
- Chevauchement : aucun pour cette première version.

#### **Fichiers produits**

- `data/corpus/chunks/chunks.jsonl`
- `data/corpus/chunk_manifest.json`

#### **Résultat attendu**

Un corpus de chunks traçables, avec des identifiants
uniques et une configuration de découpage enregistrée.

La qualité des chunks sera étudiée dans la section 5.5.

In [ ]:
import hashlib
import json
from collections import Counter

# 1. Chargement du registre
with registry_path.open('r', encoding="utf-8") as file:
    registry = json.load(file)

all_chunks = []
total_assigned_blocks = 0
source_checksums = {}

print("[INFO] Documents à traiter :", len(registry["sources"]))

# 2. G2nération des chunks de chaque document
for source in registry["sources"]:
    source_id = source["source_id"]
    document_path = project_dir / source["processed_path"]
    document_bytes = document_path.read_bytes()

    # Intégrité
    checksum = hashlib.sha256(document_bytes).hexdigest()
    assert checksum == source["processed_sha256"], f"Intégrité incorrecte : {source_id}"

    document_text = document_bytes.decode('utf-8')

    # Utiliser la fonction validée en 5.3
    document_chunks, assigned_blocks = create_document_chunks(source, document_text, target_chunk_size)
    assert document_chunks, f"Aucun chunk généré : {source_id}"

    # Vérifier la conservation des blocs Python
    original_code_count = document_text.count("```python")
    chunk_code_count = sum(chunk["text"].count("```python") for chunk in document_chunks)
    assert original_code_count == chunk_code_count, f"Blocs de code perdus : {source_id}"

    all_chunks.extend(document_chunks)
    total_assigned_blocks += assigned_blocks

    source_checksums[source_id] = checksum

    print(f"[OK] {source_id} : {len(document_chunks)} chunks")


# 3. Vérif du corpus complet
chunk_ids = [chunk["chunk_id"] for chunk in all_chunks]

assert len(chunk_ids) == len(set(chunk_ids)), "Identifiants de chunks dupliqués"
assert all(0 < chunk["characters"] <= target_chunk_size for chunk in all_chunks), "Taille de chunk invalide"
assert all(chunk["characters"] == len(chunk["text"]) for chunk in all_chunks), "Longueur de chunk incohérente."
assert all(chunk["text"].count("```") % 2 == 0 for chunk in all_chunks), "Bloc de code incomplet."

print("[OK] Contrôles du corpus réussis.")

# 4. Préparation du contenu JSONL
jsonl_content = "".join(
    json.dumps(chunk, ensure_ascii=False, sort_keys=True) + "\n" for chunk in all_chunks
)

jsonl_bytes = jsonl_content.encode("utf-8")
chunks_checksum = hashlib.sha256(jsonl_bytes).hexdigest()

# 5. Définition des chemins de sortie
chunks_path = chunks_dir / "chunks.jsonl"
manifest_path = corpus_dir / "chunk_manifest.json"

# 6. Préparation du manifeste
manifest = {
    "schema_version": 1,
    "strategy": "markdown_sections_greedy_atomic",
    "max_chars": target_chunk_size,
    "overlap_chars": 0,
    "source_count": len(registry["sources"]),
    "chunk_count": len(all_chunks),
    "chunks_sha256": chunks_checksum,
    "source_checksums": source_checksums
}

# 7. Protection des fichiers déjà existants
if manifest_path.exists() and not chunks_path.exists():
    raise RuntimeError("Manifeste présent mais fichier JSONL absent.")

if chunks_path.exists():
    assert manifest_path.exists(), "Fichier JSONL mais sans manifeste."

    previous_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    existing_bytes = chunks_path.read_bytes()
    existing_checksum = hashlib.sha256(existing_bytes).hexdigest()
    assert existing_checksum == previous_manifest["chunks_sha256"], "Le fichier JSONL existant a été modifié."

    if existing_bytes == jsonl_bytes:
        print("[INFO] Corpus JSONL déjà à jour.")
    else:
        chunks_path.write_bytes(jsonl_bytes)
        print("[OK] Corpus JSONL régénéré.")
else:
    chunks_path.write_bytes(jsonl_bytes)
    print("[OK] Corpus JSONL créé.")

# 8. Sauvegarde du manifeste s'il a changé
manifest_content = json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True) + "\n"

if(
    not manifest_path.exists() 
    or manifest_path.read_text(encoding="utf-8") != manifest_content
):
    manifest_path.write_text(manifest_content, encoding="utf-8")
    print("[OK] Manifeste sauvegardé.")

else:
    print("[INFO] Manifeste déjà à jour.")

# 9. Vérif du fichier JSON sauvegardé
saved_chunks = [
    json.loads(line) for line in chunks_path.read_text(encoding="utf-8").splitlines() if line.strip()
]
assert len(saved_chunks) == len(all_chunks)

# 10. Affichage du bilan
topic_counts = Counter(chunk["topic"] for chunk in all_chunks)

print("\n=== Bilan du chunking ===")

print("Documents traités :", len(registry["sources"]))
print("Blocs affectés :", total_assigned_blocks)
print("Chunks générés :", len(all_chunks))

print("Plus grand chunk :", max(chunk["characters"] for chunk in all_chunks), "caractères")

print("\n=== Chunks par domaine ===")
for topic in sorted(topic_counts):
    print(f"{topic} : {topic_counts[topic]}")

print("\nFichier :", chunks_path.relative_to(project_dir))
print("Manifeste :", manifest_path.relative_to(project_dir))
print("SHA-256 JSONL :", chunks_checksum)

print("\n[OK] Corpus de chunks sauvegardé.")



### *5.5. Contrôle qualité du corpus de chunks*

#### **Objectif**

Vérifier que le découpage a conservé les informations
documentaires et produit des chunks exploitables.

#### **Contrôles**

1. Vérifier l'intégrité du fichier JSONL avec SHA-256.
2. Vérifier la cohérence avec le manifeste.
3. Contrôler les identifiants et les métadonnées.
4. Comparer les blocs originaux aux blocs des chunks.
5. Vérifier la conservation des exemples Python.
6. Mesurer la distribution des tailles des chunks.
7. Identifier les chunks particulièrement courts.

#### **Critères de validation**

- Aucun bloc élémentaire perdu ou modifié.
- Aucun identifiant dupliqué.
- Aucun chunk dépassant 1 500 caractères.
- Tous les blocs Python conservés.
- Empreintes SHA-256 cohérentes.

Les chunks courts seront signalés pour inspection,
mais ne seront pas supprimés automatiquement.

#### **Résultat attendu**

Un rapport de contrôle qualité des 62 chunks.

Aucune modification des fichiers du corpus.

In [ ]:
import hashlib
import json
from statistics import mean, median

# 1. Fichiers produits en 5.4
chunks_path = chunks_dir / "chunks.jsonl"
manifest_path = corpus_dir / "chunk_manifest.json"

assert chunks_path.is_file()
assert manifest_path.is_file()

with manifest_path.open('r', encoding="utf-8") as file:
    manifest = json.load(file)

with registry_path.open('r', encoding="utf-8") as file:
    registry = json.load(file)

chunks_bytes = chunks_path.read_bytes()

chunks = [
    json.loads(line) for line in chunks_bytes.decode('utf-8').splitlines() if line.strip()
]

# 2. Vérif de la cohérence avec le manifeste
checksum = hashlib.sha256(chunks_bytes).hexdigest()

assert checksum == manifest["chunks_sha256"]
assert len(chunks) == manifest["chunk_count"]
assert len(registry["sources"]) == manifest["source_count"]
assert manifest["max_chars"] == target_chunk_size

print("[OK] JSONL & manifeste cohérents.")

# 3. Vérif des identifiants
chunk_ids = [chunk["chunk_id"] for chunk in chunks]
assert len(chunk_ids) == len(set(chunk_ids))

print("[OK] Identifiants uniques.")

# 4. Comparaison des blocs originaux aux blocs des chunks
total_original_blocks = 0
total_recovered_blocks = 0

total_original_code = 0
total_chunk_code = 0


for source in registry["sources"]:
    source_id = source["source_id"]

    document_path = project_dir / source["processed_path"]
    document_bytes = document_path.read_bytes()

    # Vérifier que le document correspond au registre.
    document_checksum = hashlib.sha256(document_bytes).hexdigest()

    assert document_checksum == source["processed_sha256"]
    assert document_checksum == manifest["source_checksums"][source_id]

    document_text = document_bytes.decode("utf-8")

    # Retrouver les sections originales.
    sections = extract_sections(document_text)
    original_blocks = []

    for section in sections:
        heading = f"{'#' * section['level']} {section['title']}"

        section_text = section["text"].strip()
        assert section_text.startswith(heading)

        body_text = section_text[len(heading):].strip()
        if body_text:
            original_blocks.extend(extract_atomic_blocks(body_text))

    # Retrouver les chunks de ce document dans leur ordre de sauvegarde.
    document_chunks = [chunk for chunk in chunks if chunk["document_id"] == source_id]
    assert document_chunks, f"Aucun chunk pour {source_id}"

    recovered_blocks = []

    for chunk in document_chunks:

        # Vérifier les métadonnées de provenance.
        assert chunk["topic"] == source["topic"]
        assert chunk["source_url"] == source["source_url"]
        assert chunk["document_title"] == source["title"]

        # Retrouver la section associée au chunk.
        section_index = chunk["section_index"]
        assert 1 <= section_index <= len(sections)

        section = sections[section_index - 1]
        assert chunk["section"] == section["title"]

        heading = f"{'#' * section['level']} {section['title']}"
        prefix = heading + "\n\n"
        assert chunk["text"].startswith(prefix), f"Titre incorrect : {chunk['chunk_id']}"

        # Retirer le titre répété pour examiner uniquement les blocs documentaires.
        chunk_body = chunk["text"][len(prefix):]
        recovered_blocks.extend(extract_atomic_blocks(chunk_body))

        # Vérifier la taille réellement sauvegardée.
        assert chunk["characters"] == len(chunk["text"])
        assert 0 < chunk["characters"] <= target_chunk_size

    # La comparaison porte sur le contenu et son ordre.
    assert original_blocks == recovered_blocks, f"Blocs perdus, modifiés ou déplacés : {source_id}"

    # Vérifier les exemples de code Python.
    original_code_count = document_text.count("```python")
    chunk_code_count = sum(chunk["text"].count("```python") for chunk in document_chunks)
    assert original_code_count == chunk_code_count, f"Blocs Python perdus : {source_id}"

    total_original_blocks += len(original_blocks)
    total_recovered_blocks += len(recovered_blocks)

    total_original_code += original_code_count
    total_chunk_code += chunk_code_count

    print(f"[OK] {source_id} : {len(original_blocks)} blocs conservés")


# 5. Examen de la distribution des tailles.
chunk_lengths = [chunk["characters"] for chunk in chunks]

# Ce seuil sert uniquement à repérer les petits chunks.
small_chunk_threshold = 200
small_chunks = [chunk for chunk in chunks if chunk["characters"] < small_chunk_threshold]


# 6. Affichage du bilan global.
print("\n=== Bilan qualité des chunks ===")

print("Documents :", len(registry["sources"]))
print("Chunks :", len(chunks))

print("Blocs conservés :", total_recovered_blocks, "/", total_original_blocks)
print("Blocs Python conservés :", total_chunk_code, "/", total_original_code)

print("Taille minimale :", min(chunk_lengths))
print("Taille moyenne :", round(mean(chunk_lengths), 1))
print("Taille médiane :", median(chunk_lengths))
print("Taille maximale :", max(chunk_lengths))

print("Chunks < 200 caractères :", len(small_chunks))


# 7. Affichage des cinq plus petits chunks.
print("\n=== Plus petits chunks ===")
print("\n=== Analyse des chunks courts ===")

for chunk in sorted(small_chunks, key=lambda item: item["characters"]):

    print("\n" + "=" * 50)

    print("ID :", chunk["chunk_id"])
    print("Section :", chunk["section"])
    print("Taille :", chunk["characters"])

    print("\nTexte complet :")
    print(chunk["text"])

print("\n[OK] Audit du chunking terminé.")

### *5.6. Sélection des chunks destinés à la recherche*

#### **Objectif**

Créer une version du corpus adaptée à la recherche
documentaire, sans modifier le fichier JSONL original.

#### **Politique de sélection V1**

Exclure :
- les sections `Additional Resources` ;
- les sections `Exercises` ;
- les chunks contenant uniquement des liens de navigation
  `Prev Tutorial` ou `Next Tutorial`.

Conserver provisoirement :
- les sections `Goal` et `Goals` ;
- les explications techniques, même courtes ;
- les formules et exemples Python.

#### **Méthode**

1. Charger et vérifier le corpus original.
2. Examiner chaque chunk.
3. Appliquer les règles de sélection.
4. Enregistrer le corpus filtré.
5. Conserver un manifeste de sélection.
6. Vérifier la reproductibilité du résultat.

#### **Fichiers produits**

- `data/corpus/chunks/retrieval_chunks.jsonl`
- `data/corpus/retrieval_manifest.json`

#### **Résultat attendu**

Un corpus de recherche contenant uniquement les chunks
retenus par notre politique V1.

Le fichier `chunks.jsonl` restera inchangé.

Cette sélection devra être évaluée lors des tests
de recherche : elle ne garantit pas à elle seule
une meilleure pertinence.

In [ ]:
import hashlib
import json
import re
from collections import Counter

# 1. Chargement du corpus original et son amnifeste
chunks_path = chunks_dir / "chunks.jsonl"
manifest_path = corpus_dir / "chunk_manifest.json"

chunks_bytes = chunks_path.read_bytes()

with manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

assert hashlib.sha256(chunks_bytes).hexdigest() == manifest["chunks_sha256"]

original_chunks = [
    json.loads(line) for line in chunks_bytes.decode("utf-8").splitlines() if line.strip()
]

print("[OK] Corpus original vérifié.")

# 2. Définition des règles de sélection
def get_exclusion_reason(chunk):
    section_name = chunk["section"].strip().casefold()

    # Exclure les références bibliographiques
    if section_name == "additional resources":
        return "additional resources"

    # Exclure les exercices proposés par les tutoriels
    if section_name == "exercises":
        return "exercises"

    # Isoler le corps du chunkn sans son titre
    parts = chunk['text'].split("\n\n", maxsplit=1)

    body_lines = []

    if len(parts) == 2:
        body_lines = [
            line.strip() for line in parts[1].splitlines() if line.strip()
        ]

    # Exclure uniquement les chunks constitués exclusivement de liens de navigation
    navigation_only = (
        bool(body_lines) and all(re.match(r"^(prev|next)\s+tutorial\s*:", line, flags=re.IGNORECASE) for line in body_lines)
    )

    if navigation_only:
        return "navigation"

    return None

# 3. Sélection des chunks sans modif des originaux
retrieval_chunks = []
excluded_chunks = []

for chunk in original_chunks:
    reason = get_exclusion_reason(chunk)
    if reason is None:
        retrieval_chunks.append(chunk)
    else:
        excluded_chunks.append({
            "chunk_id": chunk['chunk_id'],
            "reason": reason
        })

# 4. Vérif de la cohérence de la sélection
assert len(retrieval_chunks) + len(excluded_chunks) == len(original_chunks)

original_ids = {chunk['chunk_id'] for chunk in original_chunks}
selected_ids = [chunk['chunk_id'] for chunk in retrieval_chunks]

assert len(selected_ids) == len(set(selected_ids)), "Identifiants de chunks dupliqués."
assert set(selected_ids).issubset(original_ids)
assert retrieval_chunks, "Aucun chunk retenu."

print("[OK] Sélection en mémoire validée.")

# 5. Préparation du fichier JSONL filtré
retrieval_content = "".join(
    json.dumps(chunk, ensure_ascii=False, sort_keys=True) + "\n" for chunk in retrieval_chunks
)

retrieval_bytes = retrieval_content.encode("utf-8")
retrieval_checksum = hashlib.sha256(retrieval_bytes).hexdigest()

# 6. Préparation des chemins et du manifeste
retrieval_path = chunks_dir / "retrieval_chunks.jsonl"
retrieval_manifest_path = corpus_dir / "retrieval_manifest.json"

retrieval_manifest = {
    "schema_version": 1,
    "selection_policy": "retrieval_filter_v1",
    "original_chunks_sha256": manifest["chunks_sha256"],
    "retrieval_chunks_sha256": retrieval_checksum,
    "original_count": len(original_chunks),
    "selected_count": len(retrieval_chunks),
    "excluded_count": len(excluded_chunks),
    "excluded_chunks": excluded_chunks
}

# 7. Protection des versions déjà enregistrées
if retrieval_path.exists() != retrieval_manifest_path.exists():
    raise RuntimeError("Fichier de recherche et manifeste incohérents.")

if retrieval_path.exists():
    with retrieval_manifest_path.open("r", encoding="utf-8") as file:
        previous_manifest = json.load(file)

    previous_checksum = hashlib.sha256(retrieval_path.read_bytes()).hexdigest()
    assert previous_checksum == previous_manifest["retrieval_chunks_sha256"], "Corpus de recherche modifié hors du pipeline."
    if retrieval_path.read_bytes() == retrieval_bytes:
        print("[INFO] Corpus de recherche déjà à jour")
    else:
        retrieval_path.write_bytes(retrieval_bytes)
        print("[OK] Corpus de recherche régénéré.")

else:
    retrieval_path.write_bytes(retrieval_bytes)
    print("[OK] Corpus de recherche créé.")

# 8. Sauvegarde du manifeste de sélection
manifest_content = json.dumps(retrieval_manifest, ensure_ascii=False, sort_keys=True, indent=2) + "\n"

if(not retrieval_manifest_path.exists() or retrieval_manifest_path.read_text(encoding="utf-8") != manifest_content):
    retrieval_manifest_path.write_text(manifest_content, encoding='utf-8')
    print("[OK] Manifeste de sélection sauvegardé.")
else:
    print("[INFO] Manifeste de sélection déjà à jour.")

# 9. Affichage du bilan de la sélection
reason_counts = Counter(item["reason"] for item in excluded_chunks)

print("\n=== Sélection pour le RAG ===")

print("Chunks originaux :", len(original_chunks))
print("Chunks retenus :", len(retrieval_chunks))
print("Chunks exclus :", len(excluded_chunks))

print("\n=== Motifs d'exclusion ===")

for reason, count in sorted(reason_counts.items()):
    print(f"{reason} : {count}")

print("\n=== Chunks exclus ===")

for item in excluded_chunks:
    print(f"{item['chunk_id']} : {item['reason']}")

print("\nSHA-256 :", retrieval_checksum)
print("[OK] Sélection documentaire terminée.")


## *6. Bilan final et reproductibilité du corpus*

### **Objectif**

Vérifier la cohérence finale du corpus documentaire
et produire un rapport récapitulatif reproductible.

### **Ressources produites**

- Sept documents HTML originaux.
- Sept documents Markdown nettoyés.
- Un registre documentaire avec empreintes SHA-256.
- Un corpus original de 62 chunks.
- Un corpus de recherche de 55 chunks.
- Deux manifestes décrivant le chunking et la sélection.

### **Méthode**

1. Vérifier l'intégrité des documents sources et traités.
2. Vérifier les deux fichiers JSONL et leurs manifestes.
3. Contrôler la conservation des identifiants.
4. Vérifier la correspondance entre les chunks retenus
   et les chunks exclus.
5. Calculer les statistiques finales.
6. Enregistrer un rapport JSON reproductible.

### **Fichier produit**

`data/corpus/corpus_report.json`

### **Limites**

- Le corpus couvre sept pages OpenCV seulement.
- Les illustrations ne sont pas intégrées au corpus textuel.
- Le chunking repose sur une taille en caractères.
- Les références bibliographiques et exercices sont exclus
  de la sélection V1.
- La qualité du retrieval n'a pas encore été évaluée.
- Le modèle Qwen3-1.7B n'a pas encore été testé avec RAG.

### **Résultat attendu**

Un rapport vérifiable, qui pourra servir de référence
aux prochaines expériences de recherche documentaire.

In [ ]:
import hashlib
import json
from collections import Counter
from statistics import mean, median

# 1. Fonction de calcul SHA-256
def compute_sha256(file_path):
    return hashlib.sha256(file_path.read_bytes()).hexdigest()

# 2. Métadonnées du corpus
with registry_path.open("r", encoding="utf-8") as file:
    registry = json.load(file)

with (corpus_dir / "chunk_manifest.json").open("r", encoding="utf-8") as file:
    chunk_manifest = json.load(file)

with (corpus_dir / "retrieval_manifest.json").open("r", encoding="utf-8") as file:
    retrieval_manifest = json.load(file)

# 3. Vérif des docs HTML et Markdown
for source in registry["sources"]:
    source_id = source["source_id"]
    
    raw_path = project_dir / source["raw_path"]
    processed_path = project_dir / source["processed_path"]

    assert raw_path.is_file()
    assert processed_path.is_file()
    assert compute_sha256(raw_path) == source["sha256"]

    processed_checksum = compute_sha256(processed_path)
    assert processed_checksum == source["processed_sha256"]
    assert processed_checksum == chunk_manifest["source_checksums"][source_id]

print("[OK] Intégrité des 7 docs vérifiée.")

# 4. Vérif des 2 fichiers JSONL
original_path = chunks_dir / "chunks.jsonl"
retrieval_path = chunks_dir / "retrieval_chunks.jsonl"

assert compute_sha256(original_path) == chunk_manifest["chunks_sha256"]
assert compute_sha256(retrieval_path) == retrieval_manifest["retrieval_chunks_sha256"]
assert retrieval_manifest["original_chunks_sha256"] == chunk_manifest["chunks_sha256"]

print("[OK] Empreintes des deux corpus vérifiées.")

# 5. Chargement des chunks
def load_jsonl(file_path):
    with file_path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

original_chunks = load_jsonl(original_path)
retrieval_chunks = load_jsonl(retrieval_path)

assert len(original_chunks) == chunk_manifest["chunk_count"]
assert len(retrieval_chunks) == retrieval_manifest["selected_count"]

# 6. Vérif des identifiants et de la sélection
original_ids = [chunk["chunk_id"] for chunk in original_chunks]
retrieval_ids = [chunk["chunk_id"] for chunk in retrieval_chunks]
excluded_ids = [item["chunk_id"] for item in retrieval_manifest["excluded_chunks"]]

assert len(original_ids) == len(set(original_ids))
assert len(retrieval_ids) == len(set(retrieval_ids))
assert len(excluded_ids) == len(set(excluded_ids))

assert set(retrieval_ids).isdisjoint((set(excluded_ids)))

assert set(original_ids) == (set(retrieval_ids) | set(excluded_ids))
assert len(excluded_ids) == retrieval_manifest["excluded_count"]

# Vérifier que les chunks sélectionnés n'ont pas été modifiés
original_by_id = {chunk["chunk_id"]: chunk for chunk in original_chunks}

for chunk in retrieval_chunks:
    assert chunk == original_by_id[chunk["chunk_id"]]
    assert chunk["characters"] == len(chunk["text"])
    assert 0 < chunk["characters"] <= chunk_manifest["max_chars"]

print("[OK] Sélection & identifiants vérifiés.")

# 7. Statistiques finales
chunk_lengths = [chunk["characters"] for chunk in retrieval_chunks]
topic_counts = Counter(chunk["topic"] for chunk in retrieval_chunks)
exclusion_counts = Counter(item["reason"] for item in retrieval_manifest["excluded_chunks"])

# 8. Construction du rapport
report = {
    "schema_version": 1,
    "corpus_name": registry["corpus_name"],
    "opencv_version": registry["opencv_version"],
    "source_count": len(registry["sources"]),
    "original_chunk_count": len(original_chunks),
    "retrieval_chunk_count": len(retrieval_chunks),
    "excluded_chunk_count": len(excluded_ids),
    "chunking": {
        "strategy": chunk_manifest["strategy"],
        "max_chars": chunk_manifest["max_chars"],
        "overlap_chars": chunk_manifest["overlap_chars"]
    },
    "retrieval_chunk_lengths": {
        "minimum": min(chunk_lengths),
        "mean": round(mean(chunk_lengths), 1),
        "median": median(chunk_lengths),
        "maximum": max(chunk_lengths)
    },
    "retrieval_chunks_by_topic": dict(topic_counts),
    "exclusions_by_reason": dict(exclusion_counts),
    "checksums": {
        "original_chunks": chunk_manifest["chunks_sha256"],
        "retrieval_chunks": retrieval_manifest["retrieval_chunks_sha256"]
    }
}

# 9. Enregistrement du rapport
report_path = corpus_dir / "corpus_report.json"
report_content = json.dumps(report, ensure_ascii=False, indent=2, sort_keys=True) + "\n"

if report_path.exists():
     # Éviter d'écraser silencieusement un rapport modifié.
    assert report_path.read_text(encoding="utf-8") == report_content, "Le rapport existant diffère du rapport calculé."
    print("[INFO] Rapport déjà à jour.")
else:
    report_path.write_bytes(report_content.encode("utf-8"))
    print("[OK] Rapport créé.")

# 10. Affichage du bilan final
print("\n=== Bilan final du corpus ===")

print("Documents :", report["source_count"])
print("Chunks originaux :", report["original_chunk_count"])
print("Chunks RAG :", report["retrieval_chunk_count"])
print("Chunks exclus :", report["excluded_chunk_count"])

print("\n=== Répartition du corpus RAG ===")

for topic, count in sorted(topic_counts.items()):
    print(f"{topic} : {count}")

print("\n=== Taille des chunks RAG ===")

for metric, value in report["retrieval_chunk_lengths"].items():
    print(f"{metric} : {value}")

print("\nRapport :", report_path.relative_to(project_dir))

print("[OK] Bilan final du corpus terminé.")




## ***Conclusion — Construction du corpus documentaire***

### **Objectif atteint**

Nous avons construit un pipeline de préparation documentaire
destiné à alimenter un futur système RAG spécialisé en
Computer Vision et en vision industrielle.

Le corpus repose sur la documentation officielle
OpenCV 4.13.0.

### **1. Acquisition et préparation**

- 7 documents HTML téléchargés.
- 5 domaines techniques représentés.
- 7 documents convertis en Markdown.
- 42 335 caractères de texte structuré.
- 26 blocs de code Python conservés.
- Références aux illustrations absentes explicitement signalées.
- Empreintes SHA-256 enregistrées pour les documents.

### **2. Découpage documentaire**

La stratégie retenue est un découpage hiérarchique
par sections Markdown, avec regroupement progressif
des blocs élémentaires.

**Configuration :**

- Taille maximale : 1 500 caractères.
- Chevauchement : 0 caractère.
- Blocs de code et formules préservés.
- Titres de section répétés dans chaque chunk.

**Résultats :**

| Indicateur | Résultat |
|---|---:|
| Chunks originaux | 62 |
| Blocs élémentaires conservés | 228/228 |
| Blocs Python conservés | 26/26 |
| Taille maximale observée | 1 475 caractères |

### **3. Sélection documentaire pour le RAG**

Une politique de filtrage V1 a été appliquée afin
d'exclure les passages peu pertinents pour la recherche.

| Catégorie | Chunks exclus |
|---|---:|
| Additional Resources | 4 |
| Exercises | 2 |
| Navigation | 1 |
| **Total** | **7** |

Le corpus de recherche final contient **55 chunks**.

**Répartition par domaine :**

| Domaine | Chunks |
|---|---:|
| Edges | 23 |
| Filtering | 7 |
| Industrial inspection | 6 |
| Morphology | 10 |
| Segmentation | 9 |
| **Total** | **55** |

**Statistiques des chunks retenus :**

- Taille minimale : 109 caractères.
- Taille moyenne : 757,5 caractères.
- Taille médiane : 808 caractères.
- Taille maximale : 1 475 caractères.

### **4. Reproductibilité**

Les vérifications réalisées ont confirmé :

- L'intégrité des documents téléchargés et traités.
- La conservation des blocs élémentaires et des exemples Python.
- La cohérence des fichiers JSONL et de leurs manifestes.
- L'unicité des identifiants documentaires.
- L'idempotence de la sélection des chunks.
- L'idempotence de la génération du rapport final.

Les principaux fichiers produits sont :

- `data/corpus/source_registry.json`
- `data/corpus/chunks/chunks.jsonl`
- `data/corpus/chunks/retrieval_chunks.jsonl`
- `data/corpus/chunk_manifest.json`
- `data/corpus/retrieval_manifest.json`
- `data/corpus/corpus_report.json`

### **5. Limites de cette première version**

Ce corpus présente plusieurs limites :

- Il contient seulement sept documents OpenCV.
- La couverture documentaire varie entre les domaines.
- Les illustrations ne sont pas intégrées au corpus textuel.
- La taille des chunks est mesurée en caractères et non en tokens.
- Le filtrage V1 repose sur des règles documentaires simples.
- La pertinence de la recherche n'a pas encore été mesurée.
- Aucun résultat RAG avec Qwen3-1.7B n'a encore été obtenu.

Les contrôles d'intégrité ne garantissent pas à eux seuls
l'exactitude scientifique de toutes les informations extraites.

### **6. Suite du projet**

Le corpus préparé servira à :

1. Générer des représentations vectorielles (*embeddings*).
2. Construire un index de recherche documentaire.
3. Évaluer la pertinence des passages retrouvés.
4. Intégrer ces passages aux prompts de Qwen3-1.7B.
5. Comparer le RAG aux résultats du benchmark initial.

### **Conclusion générale**

Le pipeline documentaire est fonctionnel et ses sorties
ont été contrôlées.

Nous disposons désormais d'un corpus de recherche
structuré, traçable et reproductible, qui servira de base
aux prochaines expériences RAG.